# H3 directional experiments

In [ ]:
from functools import partial
from pathlib import Path
import json
import random
import shutil
import time

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
import torch_geometric
import yaml
from IPython.display import FileLink, display

from padova_core.models import DirectionalRegressor
from padova_core.neighbors import scale_directional_inputs

# Experiment settings (change RUN_DATE to run another date)
RUN_DATE = "2023-08-15"
RUN_SPLITS = [42, 123, 2026, 3407, 777]

ALL_SPLITS = [42, 123, 2026, 3407, 777]
INIT_SEEDS = [11, 22, 33]
MODELS = ["MLP_center", "MLP_ordered", "GCN", "GraphSAGE", "GraphConv", "RGCN_directional"]
FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]

N_TRIALS = 20
TUNE_MAX_EPOCHS = 1500
TUNING_INIT_SEED = 11
DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("PyTorch:", torch.__version__, "| PyG:", torch_geometric.__version__)

# Find data
PACKAGE = "h3_r10_three_dates_public_common5712_v1"
candidates = [
    p.parent for p in Path("/kaggle/input").rglob("experiment_config.json")
    if p.parent.name == PACKAGE and (p.parent / "shared" / "neighbor_index.npy").exists()
]
if not candidates:
    raise FileNotFoundError(f"Could not find {PACKAGE}")

DATA = candidates[0]
SHARED = DATA / "shared"
DAY = DATA / RUN_DATE

OUT = Path("/kaggle/working") / f"results_h3_5712_six_models_{RUN_DATE}"
OUT.mkdir(parents=True, exist_ok=True)

# Load node features, targets, and 6-direction H3 graph
X = np.load(DAY / "X_center_raw.npy").astype(np.float32)
Y = np.load(DAY / "y_lst_C.npy").astype(np.float32).reshape(-1)
NB = np.load(SHARED / "neighbor_index.npy")
E = np.load(SHARED / "edge_index.npy")
R = np.load(SHARED / "edge_type.npy")
n = len(Y)

# Check that edge types match neighbor slots
src, dst = E
assert np.array_equal(NB[dst, R], src), "Edge types do not match neighbor slots."

edge_index = torch.as_tensor(E, dtype=torch.long, device=DEVICE)
edge_type = torch.as_tensor(R, dtype=torch.long, device=DEVICE)

# Load fixed 80/10/10 splits
split_summary = pd.read_csv(SHARED / "split_summary.csv").set_index("split_seed")
MASKS = {}
for split in ALL_SPLITS:
    with np.load(SHARED / "splits" / f"split_{split}.npz") as f:
        masks = {k: f[k].astype(bool) for k in f.files}
    assert np.all(masks["train_mask"].astype(int) + masks["val_mask"].astype(int) + masks["test_mask"].astype(int) == 1)
    assert np.array_equal(masks["final_train_mask"], masks["train_mask"] | masks["val_mask"])
    MASKS[split] = masks

display(split_summary.loc[RUN_SPLITS])

## Scaling, model wrapper, and training helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def to_tensor(arr):
    return torch.as_tensor(arr, dtype=torch.float32, device=DEVICE)

def mask_to_idx(mask):
    return torch.as_tensor(np.flatnonzero(mask), dtype=torch.long, device=DEVICE)

# Standardize center and 6-direction neighbor features using fit_mask only
def prepare_data(fit_mask):
    center, ordered, target, scaler = scale_directional_inputs(X, Y, fit_mask, NB)
    return {
        "center": to_tensor(center),
        "ordered": to_tensor(ordered),
        "y": to_tensor(target),
        "scaler": scaler,
        "y_mean": scaler["y_mean"],
        "y_std": scaler["y_std"],
    }

# Bind fixed graph arguments to DirectionalRegressor
LSTModel = partial(
    DirectionalRegressor,
    n_features=len(FEATURES),
    n_directions=NB.shape[1],
    edge_index=edge_index,
    edge_type=edge_type,
    dropout=DROPOUT,
)

def make_model(name, params, seed):
    set_seed(seed)
    model = LSTModel(name, params).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)
    return model, optimizer

###########################
def train_epoch(model, optimizer, data, fit_idx):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    pred = model(data)
    loss = F.mse_loss(pred[fit_idx], data["y"][fit_idx])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    return loss.detach()

# Save experiment protocol
protocol = {
    "date": RUN_DATE,
    "graph": "H3 resolution 10; common nodes across three dates",
    "nodes": n,
    "undirected_edges": int(E.shape[1] // 2),
    "features": FEATURES,
    "models": MODELS,
    "split_seeds": ALL_SPLITS,
    "init_seeds": INIT_SEEDS,
    "tuning_init_seed": TUNING_INIT_SEED,
    "trials_per_model_per_split": N_TRIALS,
    "max_tuning_epochs": TUNE_MAX_EPOCHS,
    "final_epochs": "best validation epoch of selected trial",
    "optimizer": "AdamW",
    "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip": GRAD_CLIP,
}
with open(OUT / "experiment_config.yml", "w", encoding="utf-8") as f:
    yaml.safe_dump(protocol, f, sort_keys=False, allow_unicode=True)

## Optuna hyperparameter tuning and final refit on train + val

In [ ]:
def tune_one(name, split, data, tr_idx, va_idx, folder):
    best_file = folder / "best_params.json"
    if best_file.exists():
        return read_json(best_file)

    study = optuna.create_study(
        study_name=f"h3_{RUN_DATE}_{name}_{split}",
        storage=f"sqlite:///{folder / 'optuna.db'}",
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=split),
        load_if_exists=True,
    )

    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "n_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
            "activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu"]),
            "patience": trial.suggest_categorical("patience", [50, 100, 150]),
        }

        model, optimizer = make_model(name, params, TUNING_INIT_SEED)
        best_rmse = float("inf")
        best_epoch = 0
        history = []

        for epoch in range(1, TUNE_MAX_EPOCHS + 1):
            loss = train_epoch(model, optimizer, data, tr_idx)

            model.eval()
            with torch.no_grad():
                pred = model(data)[va_idx]
                val_rmse = (pred - data["y"][va_idx]).square().mean().sqrt().item() * data["y_std"]

            history.append((epoch, loss.item(), val_rmse))
            if val_rmse < best_rmse:
                best_rmse = val_rmse
                best_epoch = epoch

            if epoch - best_epoch >= params["patience"]:
                break

        trial.set_user_attr("best_epoch", best_epoch)
        trial.set_user_attr("last_epoch", epoch)

        hist_dir = folder / "trial_histories"
        hist_dir.mkdir(exist_ok=True)
        pd.DataFrame(history, columns=["epoch", "train_MSE_standardized", "val_RMSE_C"]).to_csv(
            hist_dir / f"trial_{trial.number}.csv", index=False
        )

        print(f"[Optuna] {RUN_DATE} | {name} | split={split} | trial={trial.number} | val={best_rmse:.4f} C | best_epoch={best_epoch}")
        return best_rmse

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done)

    study.trials_dataframe().to_csv(folder / "optuna_trials.csv", index=False)

    best = study.best_trial
    result = {
        "date": RUN_DATE,
        "model": name,
        "split_seed": split,
        "best_trial": best.number,
        "best_epoch": int(best.user_attrs["best_epoch"]),
        "best_val_RMSE_C": float(best.value),
        "params": best.params,
    }
    shutil.copy2(folder / "trial_histories" / f"trial_{best.number}.csv", folder / "best_validation_history.csv")
    write_json(best_file, result)
    return result

def run_complete(run_dir):
    return all((run_dir / f).exists() for f in ["model.pt", "history.csv", "completed.json"])

# Refit on combined train+val nodes for the exact best_epoch found during tuning
def final_fit(name, split, best, data, fit_idx, folder, seed):
    run_dir = folder / f"init_{seed}"
    run_dir.mkdir(exist_ok=True)
    if run_complete(run_dir):
        return

    model, optimizer = make_model(name, best["params"], seed)
    start = time.perf_counter()
    losses = []

    for _ in range(best["best_epoch"]):
        loss = train_epoch(model, optimizer, data, fit_idx)
        losses.append(loss.item())

    torch.save(model.state_dict(), run_dir / "model.pt")
    pd.DataFrame({
        "epoch": np.arange(1, len(losses) + 1),
        "trainval_MSE_standardized": losses,
    }).to_csv(run_dir / "history.csv", index=False)

    write_json(run_dir / "completed.json", {
        "date": RUN_DATE, "model": name, "split_seed": split, "init_seed": seed,
        "epochs": best["best_epoch"], "trainval_nodes": int(fit_idx.numel()),
        "parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "seconds": round(time.perf_counter() - start, 2),
    })

def backup_results():
    archive = shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)
    display(FileLink(Path(archive).name))

# Run tuning and final refit across selected splits
for split in RUN_SPLITS:
    masks = MASKS[split]
    tuning_data = prepare_data(masks["train_mask"])
    final_data = prepare_data(masks["final_train_mask"])

    tr_idx = mask_to_idx(masks["train_mask"])
    va_idx = mask_to_idx(masks["val_mask"])
    fit_idx = mask_to_idx(masks["final_train_mask"])

    split_dir = OUT / f"split_{split}"
    split_dir.mkdir(exist_ok=True)
    write_json(split_dir / "tuning_scaler.json", tuning_data["scaler"])
    write_json(split_dir / "final_scaler.json", final_data["scaler"])

    print(f"\n=== {RUN_DATE} | split={split} | train={tr_idx.numel()} | val={va_idx.numel()} | train+val={fit_idx.numel()} ===")

    for name in MODELS:
        folder = split_dir / name
        folder.mkdir(exist_ok=True)
        best = tune_one(name, split, tuning_data, tr_idx, va_idx, folder)

        for seed in INIT_SEEDS:
            final_fit(name, split, best, final_data, fit_idx, folder, seed)

    backup_results()

## Evaluate held-out test nodes and paired model comparisons

In [ ]:
def evaluate_current_date():
    # Only evaluate when all splits, models, and seeds are done
    for split in ALL_SPLITS:
        for name in MODELS:
            for seed in INIT_SEEDS:
                if not run_complete(OUT / f"split_{split}" / name / f"init_{seed}"):
                    print("Some runs are still missing. Skipping evaluation.")
                    return

    run_rows = []
    param_rows = []

    for split in ALL_SPLITS:
        masks = MASKS[split]
        data = prepare_data(masks["final_train_mask"])
        te_idx = mask_to_idx(masks["test_mask"])
        observed = Y[masks["test_mask"]].astype(np.float64)
        sst = np.sum((observed - observed.mean()) ** 2)

        for name in MODELS:
            folder = OUT / f"split_{split}" / name
            best = read_json(folder / "best_params.json")
            model = LSTModel(name, best["params"]).to(DEVICE)
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

            param_rows.append({
                "date": RUN_DATE, "model": name, "split_seed": split,
                **best["params"],
                "best_trial": best["best_trial"],
                "best_epoch": best["best_epoch"],
                "best_at_epoch_cap": best["best_epoch"] == TUNE_MAX_EPOCHS,
                "best_val_RMSE_C": best["best_val_RMSE_C"],
                "parameters": n_params,
            })

            for seed in INIT_SEEDS:
                state = torch.load(folder / f"init_{seed}" / "model.pt", map_location=DEVICE, weights_only=True)
                model.load_state_dict(state)
                model.eval()

                with torch.no_grad():
                    pred_std = model(data)[te_idx].cpu().numpy().astype(np.float64)
                pred = pred_std * data["y_std"] + data["y_mean"]
                err = pred - observed

                run_rows.append({
                    "date": RUN_DATE, "model": name, "split_seed": split, "init_seed": seed,
                    "test_nodes": len(observed),
                    "MAE": float(np.abs(err).mean()),
                    "RMSE": float(np.sqrt(np.mean(err ** 2))),
                    "R2": float(1 - np.sum(err ** 2) / sst),
                    "Bias": float(err.mean()),
                })

    runs = pd.DataFrame(run_rows)
    params_df = pd.DataFrame(param_rows)
    metrics = ["MAE", "RMSE", "R2", "Bias"]

    # Average over init seeds within each split, then mean & std across splits
    per_split = runs.groupby(["date", "model", "split_seed"], as_index=False)[metrics].mean()
    summary = per_split.groupby(["date", "model"])[metrics].agg(["mean", "std"])
    summary.columns = [f"{m}_{s}" for m, s in summary.columns]
    summary = summary.reset_index().sort_values("RMSE_mean")

    # Paired RMSE reduction (baseline - candidate) across the 5 splits
    pairs = [
        ("MLP_center", "MLP_ordered"),
        ("MLP_ordered", "GCN"),
        ("MLP_ordered", "GraphSAGE"),
        ("MLP_ordered", "GraphConv"),
        ("MLP_ordered", "RGCN_directional"),
        ("GCN", "RGCN_directional"),
        ("GraphSAGE", "RGCN_directional"),
        ("GraphConv", "RGCN_directional"),
    ]
    wide = per_split.pivot(index="split_seed", columns="model", values="RMSE")
    paired_rows = []
    for base, cand in pairs:
        for split in ALL_SPLITS:
            paired_rows.append({
                "date": RUN_DATE, "baseline": base, "candidate": cand, "split_seed": split,
                "RMSE_reduction_C": wide.loc[split, base] - wide.loc[split, cand],
            })

    paired = pd.DataFrame(paired_rows)
    paired_summary = paired.groupby(["date", "baseline", "candidate"], sort=False).agg(
        mean_RMSE_reduction_C=("RMSE_reduction_C", "mean"),
        SD_across_splits_C=("RMSE_reduction_C", "std"),
        candidate_better_splits=("RMSE_reduction_C", lambda x: int((x > 0).sum())),
        number_of_splits=("RMSE_reduction_C", "count"),
    ).reset_index()

    # Save all evaluation tables
    runs.to_csv(OUT / "test_metrics_all_runs.csv", index=False)
    per_split.to_csv(OUT / "comparison_per_split.csv", index=False)
    summary.to_csv(OUT / "comparison_summary.csv", index=False)
    params_df.to_csv(OUT / "best_params_all_splits.csv", index=False)
    paired.to_csv(OUT / "paired_differences_each_split.csv", index=False)
    paired_summary.to_csv(OUT / "paired_differences_summary.csv", index=False)

    display(summary.round(4))
    display(paired_summary.round(4))
    backup_results()

evaluate_current_date()